# 01 — Ambiente e infraestrutura (Etapa 1)

Verifica o ambiente de trabalho do [PLANO_IMPLEMENTACAO_ANOMALIA.md](../../PLANO_IMPLEMENTACAO_ANOMALIA.md):

- **Colab (GPU):** extração TerraMind e treino. Dados, embeddings, checkpoints e resultados ficam no Google Drive (`MyDrive/ProjetoGrassQuality`).
- **Local (Windows, sem GPU):** análise com numpy/matplotlib e mapas. O Smart App Control bloqueia as DLLs de pandas/scipy/sklearn, então as células de GPU são puladas e o bloqueio fica registrado.

**Entregável da Etapa 1:** `import terratorch, torch; torch.cuda.is_available()` retorna `True` no Colab (*Ambiente de execução → Alterar tipo → T4 GPU*).

## 1. Plataforma

In [ ]:
import sys, os, subprocess, time, google
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Colab' if IN_COLAB else 'Local', '| Python', sys.version.split()[0])

## 2. Setup no Colab (pulado localmente)

Monta o Drive, clona/atualiza o repositório e instala as dependências.
Se o repositório for privado, crie um segredo `GITHUB_TOKEN` em *Colab → Secrets* (ícone de chave) com um token de leitura.

In [ ]:
REPO_URL = 'github.com/GabrielPow/ProjetoGrassQuality.git'
BRANCH = 'main'
REPO_DIR = '/content/ProjetoGrassQuality'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    url = f'https://{token}@{REPO_URL}' if token else f'https://{REPO_URL}'

    if os.path.exists(REPO_DIR):
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
    else:
        subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)

    from importlib import metadata
    import numpy
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    f'{REPO_DIR}/notebooks/gabriel/requirements-colab.txt'], check=True)
    if metadata.version('numpy') != numpy.__version__:
        # o pip trocou o numpy já carregado; sem reiniciar, terratorch/sklearn falham com "_slice"
        print(f'numpy {numpy.__version__} -> {metadata.version("numpy")}: reiniciando o ambiente. '
              'Rode todas as células de novo (a instalação não se repete).')
        os.kill(os.getpid(), 9)
    os.chdir(f'{REPO_DIR}/notebooks/gabriel')
else:
    print('Local: setup do Colab pulado.')

In [ ]:
import config, utils
config.make_dirs()
print('REPO_ROOT:', config.REPO_ROOT)
print('DATA_ROOT:', config.DATA_ROOT)

## 3. Bibliotecas

In [ ]:
import importlib
from importlib import metadata

# sem pandas aqui: localmente o App Control também bloqueia as DLLs do pandas
LIBS = ['numpy', 'pandas', 'torch', 'terratorch', 'sklearn', 'scipy', 'rasterio',
        'xarray', 'zarr', 'geopandas', 'ee']
DIST = {'sklearn': 'scikit-learn', 'ee': 'earthengine-api'}  # nome do módulo -> nome do pacote
rows = []
for name in LIBS:
    try:
        importlib.import_module(name)
        rows.append({'lib': name, 'ok': True, 'versao': metadata.version(DIST.get(name, name)), 'erro': ''})
    except Exception as e:  # ImportError, ou OSError quando o App Control bloqueia a DLL
        rows.append({'lib': name, 'ok': False, 'versao': None, 'erro': f'{type(e).__name__}: {e}'[:120]})
libs = rows
for r in libs:
    print(f"{r['lib']:<11}{'OK ' if r['ok'] else 'X  '}{r['versao'] or r['erro']}")

## 4. Entregável: torch + terratorch com GPU

In [ ]:
gpu = {'cuda': False}
try:
    import terratorch, torch
    gpu['cuda'] = torch.cuda.is_available()
    print('torch', torch.__version__, '| terratorch', metadata.version('terratorch'), '| cuda:', gpu['cuda'])
    if gpu['cuda']:
        p = torch.cuda.get_device_properties(0)
        gpu.update(name=p.name, mem_gb=round(p.total_memory / 1e9, 1))
        print(gpu)
except ImportError as e:
    print('Indisponível neste ambiente:', e)

## 5. TerraMind Small: nome no registry e forward pass de teste

Resolve o item da Etapa 4, "conferir o nome exato no registry". O S2L2A do TerraMind usa 12 bandas (B1–B12 sem B10).
Com patch 224 px e tokens de 16 px, a saída esperada tem **196 tokens (14 × 14)** por camada.

In [ ]:
terramind = {}
if gpu['cuda']:
    from terratorch.registry import BACKBONE_REGISTRY

    try:
        names = sorted(n for n in BACKBONE_REGISTRY if 'terramind' in str(n).lower())
    except TypeError:
        names = []
    print('TerraMind no registry:', names)
    terramind['name'] = config.TERRAMIND_SMALL
    assert terramind['name'] in names, f"{terramind['name']} não está no registry desta versão do terratorch"

    model = BACKBONE_REGISTRY.build(terramind['name'], pretrained=True, modalities=['S2L2A'])
    model = model.eval().cuda()
    for prm in model.parameters():
        prm.requires_grad = False

    x = {'S2L2A': torch.randn(1, 12, config.PATCH_SIZE, config.PATCH_SIZE, device='cuda')}
    with torch.no_grad():
        t0 = time.time(); out = model(x); torch.cuda.synchronize(); dt = time.time() - t0
    last = out[-1] if isinstance(out, (list, tuple)) else out
    terramind.update(n_layers_out=len(out) if isinstance(out, (list, tuple)) else 1,
                     token_shape=list(last.shape), forward_s=round(dt, 3))
    print(terramind)
else:
    print('Sem GPU: teste do TerraMind pulado (rode no Colab com T4).')

## 6. Earth Engine

In [ ]:
ee_info = {}
try:
    import ee
    try:
        ee.Initialize(project=config.EE_PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=config.EE_PROJECT_ID)
    ee_info['lulc_years'] = ee.ImageCollection(config.LULC_ASSET).size().getInfo()
    print('Earth Engine OK | imagens MapBiomas LULC:', ee_info['lulc_years'])
except Exception as e:
    ee_info['error'] = f'{type(e).__name__}: {e}'[:200]
    print('Earth Engine indisponível:', ee_info['error'])

## 7. Regressão das métricas portadas

Recalcula "AE cosine change (T)", *todas as anomalias vs. todos os normais*, do first pass com as funções de `utils.py`.
Precisa bater com `experiments/amazon_anomaly_firstpass/results/metrics.csv` (AUROC 0,89).

In [ ]:
import numpy as np

d = np.load(config.FIRSTPASS_DIR / 'data' / 'amazon_firstpass.npz', allow_pickle=True)
AE, S2, years, T, strat, split, block = (d[k] for k in ['AE', 'S2', 'years', 'T', 'strat', 'split', 'block'])
rows_ = np.arange(len(T)); jT = np.searchsorted(years, T)
at = lambda arr, off: arr[rows_, jT + off]

ae_hist = np.mean([at(AE, -k) for k in (1, 2)], axis=0)
s2_hist = np.mean([at(S2, -k) for k in (1, 2)], axis=0)
ok = ~np.any([np.isnan(a).any(1) for a in
              (ae_hist, at(AE, 0), at(AE, 1), s2_hist, at(S2, 0), at(S2, 1))], axis=0)

unit = lambda v: v / np.linalg.norm(v, axis=1, keepdims=True)
score = 1 - np.sum(unit(at(AE, 0)) * unit(ae_hist), axis=1)
y = np.isin(strat, [2, 3]).astype(int)
m = ok & (split == 'test')

lo, hi = utils.block_bootstrap(block[m], y[m], score[m], utils.auroc)
check = {'AUROC': utils.auroc(y[m], score[m]), 'AUROC_lo': lo, 'AUROC_hi': hi,
         'AP': utils.avg_precision(y[m], score[m]), 'TPR_at_5FPR': utils.tpr_at_fpr(y[m], score[m], 0.05)}

import csv
with open(config.FIRSTPASS_DIR / 'results' / 'metrics.csv', encoding='utf-8') as f:
    ref = next(r for r in csv.DictReader(f)
               if r['task'] == 'all anomalies vs all normals' and r['score'] == 'AE cosine change (T)')
print(f"{'':<12}{'utils.py':>10}{'metrics.csv':>13}")
for k, v in check.items():
    print(f'{k:<12}{v:>10.4f}{float(ref[k]):>13.4f}')
assert np.allclose([check[k] for k in check], [float(ref[k]) for k in check]),     'métricas portadas divergem do first pass'
print('OK: métricas portadas reproduzem o first pass.')
print('Extras: TPR@1%FPR =', round(utils.tpr_at_fpr(y[m], score[m], 0.01), 3),
      '| precisão@100 =', utils.precision_at_k(y[m], score[m], 100))

## 8. Semente e log da execução

In [ ]:
utils.set_seed(config.SEED)
run_dir = utils.new_run('etapa1_ambiente')
import json
(run_dir / 'ambiente.json').write_text(json.dumps(
    {'bibliotecas': libs, 'gpu': gpu, 'terramind': terramind, 'earth_engine': ee_info,
     'metricas_first_pass': {k: float(v) for k, v in check.items()}}, indent=2), encoding='utf-8')
print('Salvo em', run_dir)
print(json.loads((run_dir / 'run.json').read_text())['git'])

## 9. Fixar versões (só no Colab)

Gera `requirements-colab.lock.txt` com as versões instaladas, no clone em `/content` e na pasta da execução no Drive (o clone some quando o ambiente reinicia). Copie o do Drive para o repositório e faça commit.

In [ ]:
if IN_COLAB:
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
    keep = {p.lower() for p in utils.KEY_PACKAGES} | {'torchvision', 'pyyaml', 'timm'}
    lock = [l for l in freeze.splitlines() if l.split('==')[0].lower() in keep]
    for dest in ('requirements-colab.lock.txt', run_dir / 'requirements-colab.lock.txt'):
        open(dest, 'w').write('\n'.join(lock) + '\n')
    print('\n'.join(lock))
else:
    print('Local: lock pulado.')